# Extract Shot Lengths with TransNet V2

This notebook creates a TransNet V2 comparison dataset for the same video directory used by `extract_shot_lengths.ipynb`. It keeps independent outputs in `transnet_shot_length_results/`, supports movie-level pause/resume, and can process either the whole folder or only films listed in `shot_length_results/pyscenedetect_bad_results.csv`.

Default mode processes only the bad-results list, so the first run is a small benchmark rather than a full 454-film rerun.

## TransNet V2 Setup

This notebook expects a local TransNet V2 checkout with model weights. Recommended layout:

```text
external/TransNetV2/
  inference/transnetv2.py
  inference/transnetv2-weights/
```

If you do not have it yet, clone the official repository with Git LFS enabled outside this notebook, then install the TensorFlow dependencies in the Jupyter kernel environment. The original TransNet V2 TensorFlow code is old, so environment setup may need a separate conda/venv if your current Python cannot install a compatible TensorFlow.

In [6]:
from __future__ import annotations

import json
import math
import os
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

VIDEO_DIR = Path("/Volumes/Extreme SSD/Video/Chinese1949-66")
OUTPUT_DIR = Path("./transnet_shot_length_results")

BAD_RESULTS_CSV = Path("./shot_length_results/pyscenedetect_bad_results.csv")
ONLY_PROCESS_BAD_RESULTS = True

# Use a small integer for smoke testing, e.g. MAX_MOVIES = 2. Use None for all selected movies.
MAX_MOVIES = None
ROUND_SECONDS = 3
TRANSNET_PREDICTION_THRESHOLD = 0.5

# Keep False by default because TensorFlow installs can be environment-specific.
# If dependency checks fail and you want the notebook to try installing into this kernel,
# set this to True and rerun the setup/import cell.
AUTO_INSTALL_TRANSNET_DEPENDENCIES = False

# Local checkout of https://github.com/soCzech/TransNetV2
TRANSNET_REPO_DIR = Path("./external/TransNetV2-git")
TRANSNET_INFERENCE_DIR = TRANSNET_REPO_DIR / "inference"
TRANSNET_WEIGHTS_DIR = TRANSNET_INFERENCE_DIR / "transnetv2-weights"

PROGRESS_CSV = OUTPUT_DIR / "movie_progress_transnet.csv"
LONG_CSV = OUTPUT_DIR / "shot_lengths_long_transnet.csv"
WIDE_CSV = OUTPUT_DIR / "shot_lengths_wide_transnet.csv"
COMPARISON_CSV = OUTPUT_DIR / "detector_comparison_summary.csv"

PYSCENEDETECT_PROGRESS_CSV = Path("./shot_length_results/movie_progress.csv")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MPL_CACHE_DIR = OUTPUT_DIR / "matplotlib_cache"
MPL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPL_CACHE_DIR.resolve()))

print(f"Video directory: {VIDEO_DIR}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")
print(f"Only process bad-results list: {ONLY_PROCESS_BAD_RESULTS}")

Video directory: /Volumes/Extreme SSD/Video/Chinese1949-66
Output directory: /Users/caoshuangyuan/Desktop/Workspace/DirtyPilot/transnet_shot_length_results
Only process bad-results list: True


In [7]:
def _missing_modules(module_names: list[str]) -> list[str]:
    missing = []
    for module_name in module_names:
        try:
            __import__(module_name)
        except ModuleNotFoundError:
            missing.append(module_name)
    return missing


def verify_transnet_dependencies() -> None:
    # TransNet's predict_video() imports the Python package named `ffmpeg`,
    # which is installed from the PyPI package `ffmpeg-python`.
    module_to_package = {"tensorflow": "tensorflow", "ffmpeg": "ffmpeg-python"}
    missing = _missing_modules(list(module_to_package))
    if missing and AUTO_INSTALL_TRANSNET_DEPENDENCIES:
        packages = [module_to_package[name] for name in missing]
        print("Installing missing TransNet dependencies into this Jupyter kernel:", packages)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", *packages])
        missing = _missing_modules(list(module_to_package))

    if missing:
        packages = " ".join(module_to_package[name] for name in missing)
        raise ModuleNotFoundError(
            "Missing TransNet dependency module(s): " + ", ".join(missing) + "\n\n"
            "This Jupyter kernel is running:\n"
            f"  {sys.executable}\n\n"
            "Install into this exact kernel with a notebook cell like:\n"
            f"  import sys\n  !{{sys.executable}} -m pip install -U {packages}\n\n"
            "Then restart the kernel and rerun the notebook from the top."
        )


def verify_transnet_installation() -> None:
    inference_file = TRANSNET_INFERENCE_DIR / "transnetv2.py"
    if not inference_file.exists():
        raise FileNotFoundError(
            "Cannot find TransNet V2 inference code. Expected file:\n"
            f"  {inference_file}\n\n"
            "Clone the official repo into ./external/TransNetV2 or change TRANSNET_REPO_DIR."
        )
    if not TRANSNET_WEIGHTS_DIR.exists():
        raise FileNotFoundError(
            "Cannot find TransNet V2 weights directory. Expected directory:\n"
            f"  {TRANSNET_WEIGHTS_DIR}\n\n"
            "The official repo uses model weights; make sure they were downloaded, often via Git LFS."
        )
    if str(TRANSNET_INFERENCE_DIR.resolve()) not in sys.path:
        sys.path.insert(0, str(TRANSNET_INFERENCE_DIR.resolve()))


verify_transnet_dependencies()
verify_transnet_installation()

from transnetv2 import TransNetV2

print("TransNet V2 import path looks ready.")

TransNet V2 import path looks ready.


In [8]:
def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def list_video_files(video_dir: Path) -> list[Path]:
    if not video_dir.exists():
        raise FileNotFoundError(f"Video directory does not exist: {video_dir}")
    return sorted(
        path
        for path in video_dir.iterdir()
        if path.is_file()
        and path.suffix.lower() == ".mp4"
        and not path.name.startswith("._")
    )


def load_bad_film_names(path: Path) -> set[str]:
    if not path.exists():
        raise FileNotFoundError(f"Bad-results CSV does not exist: {path}")
    bad_df = pd.read_csv(path)
    if "film" in bad_df.columns:
        series = bad_df["film"]
    else:
        series = bad_df.iloc[:, 0]
    names = {str(value).strip() for value in series.dropna() if str(value).strip()}
    if not names:
        raise ValueError(f"Bad-results CSV has no film names: {path}")
    return names


def select_video_files() -> list[Path]:
    videos = list_video_files(VIDEO_DIR)
    if not ONLY_PROCESS_BAD_RESULTS:
        return videos

    bad_names = load_bad_film_names(BAD_RESULTS_CSV)
    selected = [path for path in videos if path.stem in bad_names]
    missing = sorted(bad_names - {path.stem for path in selected})
    if missing:
        print("WARNING: These bad-list films were not found as .mp4 files:")
        for name in missing:
            print(f"  - {name}")
    return selected


def movie_metadata(video_path: Path) -> dict:
    stat = video_path.stat()
    return {
        "film": video_path.stem,
        "video_path": str(video_path),
        "file_size": int(stat.st_size),
        "mtime": int(stat.st_mtime_ns),
    }


def read_csv_or_empty(path: Path, columns: list[str]) -> pd.DataFrame:
    if path.exists() and path.stat().st_size > 0:
        return pd.read_csv(path)
    return pd.DataFrame(columns=columns)


def atomic_write_csv(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp_path, index=False, encoding="utf-8-sig")
    os.replace(tmp_path, path)


def load_progress() -> pd.DataFrame:
    columns = [
        "film", "status", "video_path", "file_size", "mtime", "shot_count",
        "error", "started_at", "finished_at", "detector", "threshold", "fps", "frame_count",
    ]
    return read_csv_or_empty(PROGRESS_CSV, columns)


def load_long_results() -> pd.DataFrame:
    columns = ["film", "shot_index", "duration_sec", "start_sec", "end_sec", "video_path"]
    return read_csv_or_empty(LONG_CSV, columns)


def replace_rows_by_film(df: pd.DataFrame, film: str, replacement: pd.DataFrame) -> pd.DataFrame:
    if not df.empty and "film" in df.columns:
        df = df[df["film"] != film].copy()
    return pd.concat([df, replacement], ignore_index=True)


def upsert_progress(progress_df: pd.DataFrame, row: dict) -> pd.DataFrame:
    return replace_rows_by_film(progress_df, row["film"], pd.DataFrame([row]))


def make_wide_results(long_df: pd.DataFrame) -> pd.DataFrame:
    if long_df.empty:
        return pd.DataFrame(columns=["film"])
    wide_rows = []
    max_shots = int(long_df["shot_index"].max())
    width = max(3, len(str(max_shots)))
    for film, group in long_df.sort_values(["film", "shot_index"]).groupby("film", sort=True):
        row = {"film": film}
        for _, shot in group.iterrows():
            shot_index = int(shot["shot_index"])
            row[f"shot_{shot_index:0{width}d}"] = shot["duration_sec"]
        wide_rows.append(row)
    return pd.DataFrame(wide_rows)


def is_complete_and_unchanged(progress_df: pd.DataFrame, metadata: dict) -> bool:
    if progress_df.empty or "film" not in progress_df.columns:
        return False
    rows = progress_df[progress_df["film"] == metadata["film"]]
    if rows.empty:
        return False
    row = rows.iloc[-1]
    if str(row.get("status", "")) != "complete":
        return False
    try:
        return (
            int(row.get("file_size", -1)) == metadata["file_size"]
            and int(row.get("mtime", -1)) == metadata["mtime"]
        )
    except Exception:
        return False

In [9]:
def parse_fps(rate: str) -> float:
    if not rate or rate == "0/0":
        return math.nan
    if "/" in rate:
        numerator, denominator = rate.split("/", 1)
        return float(numerator) / float(denominator)
    return float(rate)


def ffprobe_video_info(video_path: Path) -> dict:
    cmd = [
        "ffprobe", "-v", "error", "-select_streams", "v:0",
        "-show_entries", "stream=avg_frame_rate,r_frame_rate,nb_frames,duration",
        "-of", "json", str(video_path),
    ]
    completed = subprocess.run(cmd, check=True, capture_output=True, text=True)
    payload = json.loads(completed.stdout)
    stream = payload["streams"][0]
    fps = parse_fps(stream.get("avg_frame_rate") or stream.get("r_frame_rate"))
    if math.isnan(fps) or fps <= 0:
        fps = parse_fps(stream.get("r_frame_rate"))
    frame_count = stream.get("nb_frames")
    if frame_count is None or str(frame_count).upper() == "N/A":
        duration = float(stream.get("duration", 0.0))
        frame_count = int(round(duration * fps)) if fps > 0 else 0
    else:
        frame_count = int(frame_count)
    return {"fps": float(fps), "frame_count": int(frame_count)}


def scenes_to_long_df(video_path: Path, scenes: np.ndarray, fps: float, frame_count: int) -> pd.DataFrame:
    if scenes is None or len(scenes) == 0:
        scenes = np.array([[0, max(0, frame_count - 1)]], dtype=np.int64)

    rows = []
    for index, scene in enumerate(scenes, start=1):
        start_frame = int(scene[0])
        end_frame = int(scene[1])
        start_sec = start_frame / fps
        end_sec = (end_frame + 1) / fps
        duration_sec = max(0.0, end_sec - start_sec)
        rows.append({
            "film": video_path.stem,
            "shot_index": index,
            "duration_sec": round(duration_sec, ROUND_SECONDS),
            "start_sec": round(start_sec, ROUND_SECONDS),
            "end_sec": round(end_sec, ROUND_SECONDS),
            "video_path": str(video_path),
        })
    return pd.DataFrame(rows, columns=[
        "film", "shot_index", "duration_sec", "start_sec", "end_sec", "video_path",
    ])


transnet_model = None


def get_transnet_model():
    global transnet_model
    if transnet_model is None:
        transnet_model = TransNetV2(model_dir=str(TRANSNET_WEIGHTS_DIR))
    return transnet_model


def detect_shot_lengths_transnet(video_path: Path) -> tuple[pd.DataFrame, dict]:
    info = ffprobe_video_info(video_path)
    model = get_transnet_model()
    video_frames, single_frame_predictions, all_frame_predictions = model.predict_video(str(video_path))
    scenes = model.predictions_to_scenes(single_frame_predictions, threshold=TRANSNET_PREDICTION_THRESHOLD)
    movie_long_df = scenes_to_long_df(video_path, scenes, info["fps"], info["frame_count"])
    return movie_long_df, info

In [10]:
def write_comparison_summary(progress_df: pd.DataFrame) -> None:
    if not PYSCENEDETECT_PROGRESS_CSV.exists():
        return
    pyscene_df = pd.read_csv(PYSCENEDETECT_PROGRESS_CSV)
    if progress_df.empty:
        comparison_df = pd.DataFrame()
    else:
        comparison_df = progress_df[["film", "status", "shot_count"]].rename(
            columns={"status": "transnet_status", "shot_count": "transnet_shot_count"}
        )
        pyscene_small = pyscene_df[["film", "status", "shot_count"]].rename(
            columns={"status": "pyscenedetect_status", "shot_count": "pyscenedetect_shot_count"}
        )
        comparison_df = comparison_df.merge(pyscene_small, on="film", how="left")
        comparison_df["shot_count_delta"] = (
            comparison_df["transnet_shot_count"] - comparison_df["pyscenedetect_shot_count"]
        )
        comparison_df["shot_count_ratio"] = (
            comparison_df["transnet_shot_count"] / comparison_df["pyscenedetect_shot_count"]
        )
    atomic_write_csv(comparison_df, COMPARISON_CSV)


def write_all_outputs(progress_df: pd.DataFrame, long_df: pd.DataFrame) -> None:
    progress_df = progress_df.sort_values("film", kind="stable").reset_index(drop=True)
    long_df = long_df.sort_values(["film", "shot_index"], kind="stable").reset_index(drop=True)
    atomic_write_csv(progress_df, PROGRESS_CSV)
    atomic_write_csv(long_df, LONG_CSV)
    atomic_write_csv(make_wide_results(long_df), WIDE_CSV)
    write_comparison_summary(progress_df)


def process_movies(max_movies: int | None = MAX_MOVIES) -> tuple[pd.DataFrame, pd.DataFrame]:
    videos = select_video_files()
    progress_df = load_progress()
    long_df = load_long_results()

    candidates = []
    skipped = 0
    for video_path in videos:
        metadata = movie_metadata(video_path)
        if is_complete_and_unchanged(progress_df, metadata):
            skipped += 1
            continue
        candidates.append((video_path, metadata))

    if max_movies is not None:
        candidates = candidates[:max_movies]

    print(f"Selected {len(videos)} real .mp4 files; skipped {skipped} already complete unchanged movies.")
    print(f"Processing {len(candidates)} movie(s) this run. MAX_MOVIES={max_movies}")

    for video_path, metadata in tqdm(candidates, desc="Movies"):
        film = metadata["film"]
        started_at = utc_now_iso()
        processing_row = {
            **metadata,
            "status": "processing",
            "shot_count": 0,
            "error": "",
            "started_at": started_at,
            "finished_at": "",
            "detector": "TransNet V2",
            "threshold": TRANSNET_PREDICTION_THRESHOLD,
            "fps": math.nan,
            "frame_count": 0,
        }
        progress_df = upsert_progress(progress_df, processing_row)
        write_all_outputs(progress_df, long_df)

        try:
            movie_long_df, info = detect_shot_lengths_transnet(video_path)
            long_df = replace_rows_by_film(long_df, film, movie_long_df)
            complete_row = {
                **metadata,
                "status": "complete",
                "shot_count": int(len(movie_long_df)),
                "error": "",
                "started_at": started_at,
                "finished_at": utc_now_iso(),
                "detector": "TransNet V2",
                "threshold": TRANSNET_PREDICTION_THRESHOLD,
                "fps": info["fps"],
                "frame_count": info["frame_count"],
            }
            progress_df = upsert_progress(progress_df, complete_row)
        except Exception as exc:
            failed_row = {
                **metadata,
                "status": "failed",
                "shot_count": 0,
                "error": repr(exc),
                "started_at": started_at,
                "finished_at": utc_now_iso(),
                "detector": "TransNet V2",
                "threshold": TRANSNET_PREDICTION_THRESHOLD,
                "fps": math.nan,
                "frame_count": 0,
            }
            progress_df = upsert_progress(progress_df, failed_row)
            print(f"FAILED: {film}: {exc!r}")

        write_all_outputs(progress_df, long_df)

    return progress_df, long_df


progress_df, long_df = process_movies(MAX_MOVIES)

Selected 7 real .mp4 files; skipped 1 already complete unchanged movies.
Processing 6 movie(s) this run. MAX_MOVIES=None


Movies:   0%|          | 0/6 [00:00<?, ?it/s]

[TransNetV2] Extracting frames from /Volumes/Extreme SSD/Video/Chinese1949-66/以革命的名义.mp4
[TransNetV2] Processing video frames 222750/222760

Movies:  17%|█▋        | 1/6 [28:39<2:23:18, 1719.61s/it]

[TransNetV2] Processing video frames 222760/222760
[TransNetV2] Extracting frames from /Volumes/Extreme SSD/Video/Chinese1949-66/农奴.mp4
[TransNetV2] Processing video frames 132500/132537

Movies:  33%|███▎      | 2/6 [44:22<1:24:11, 1262.81s/it]

[TransNetV2] Processing video frames 132537/132537
[TransNetV2] Extracting frames from /Volumes/Extreme SSD/Video/Chinese1949-66/冰山上的来客.mp4
[TransNetV2] Processing video frames 152250/152272

Movies:  50%|█████     | 3/6 [1:03:14<1:00:09, 1203.03s/it]

[TransNetV2] Processing video frames 152272/152272
[TransNetV2] Extracting frames from /Volumes/Extreme SSD/Video/Chinese1949-66/冲破黎明前的黑暗.mp4
[TransNetV2] Processing video frames 400/140126

Movies:  50%|█████     | 3/6 [1:03:48<1:03:48, 1276.32s/it]


KeyboardInterrupt: 

In [ ]:
selected_videos = select_video_files()
progress_df = load_progress()
long_df = load_long_results()
wide_df = make_wide_results(long_df)

complete_count = int((progress_df.get("status", pd.Series(dtype=str)) == "complete").sum())
failed_count = int((progress_df.get("status", pd.Series(dtype=str)) == "failed").sum())
processing_count = int((progress_df.get("status", pd.Series(dtype=str)) == "processing").sum())
remaining_count = max(0, len(selected_videos) - complete_count)

print(f"Selected .mp4 files: {len(selected_videos)}")
print(f"Complete: {complete_count}")
print(f"Failed: {failed_count}")
print(f"Processing/incomplete from an interrupted run: {processing_count}")
print(f"Remaining not complete: {remaining_count}")
print(f"Progress CSV: {PROGRESS_CSV.resolve()}")
print(f"Long CSV: {LONG_CSV.resolve()}")
print(f"Wide CSV: {WIDE_CSV.resolve()}")
print(f"Comparison CSV: {COMPARISON_CSV.resolve()}")

display(progress_df.tail(20))
display(long_df.head(20))
if COMPARISON_CSV.exists():
    display(pd.read_csv(COMPARISON_CSV).sort_values("film").head(20))

## Full Folder Mode

After the bad-results benchmark looks useful, change `ONLY_PROCESS_BAD_RESULTS = False` in the config cell to process the whole folder. Existing complete TransNet rows will still be skipped if file size and mtime match.